# 05 — Post-training Analysis

End-to-end analysis of all trained models on the Russian annotated test set.
Builds on the raw evaluation results from `04_evaluation.ipynb` and digs into
statistical validity, embedding-space diagnostics, ensemble behaviour,
calibration, and failure modes.

**Sections**
1. Bootstrap Confidence Intervals & Statistical Tests
2. Reverse Density: Russian Coverage under ToxiGen
3. Hyperparameter Ablation: K and Embedding Space
4. Ensemble Normalisation Sweep
5. Cross-group Specialist Evaluation Matrix
6. Calibration Analysis
7. Qualitative Error Analysis

> **Prerequisites** — run notebooks 01 → 04 (including `03_01` for specialist models) before this one.

In [ ]:
import sys, os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.calibration import calibration_curve
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score
from sentence_transformers import SentenceTransformer

from src.evaluation import compute_metrics, bootstrap_ci, discover_models
from src.evaluation.comparator import _get_probabilities
from src.evaluation.ensemble import evaluate_ensemble, compute_group_weights, normalise_weights

## 0. Configuration

In [ ]:
# ── Palette options ──────────────────────────────────────────────────────────
# Each palette defines: sequential cmap, diverging cmap, categorical palette,
# a highlight colour (density-weighted models), a neutral colour (baselines),
# and explicit hate / no-hate colours for binary comparisons.

PALETTES = {
    "A": {
        "name":         "Academic — colorblind-safe, print-friendly (thesis default)",
        "sequential":   "Blues",
        "diverging":    "RdYlBu",
        "categorical":  "colorblind",
        "highlight":    "#2166ac",
        "neutral":      "#878787",
        "hate_color":   "#d6604d",
        "no_hate_color":"#4393c3",
    },
    "B": {
        "name":         "Warm editorial — good for slides and presentations",
        "sequential":   "YlOrRd",
        "diverging":    "RdYlGn",
        "categorical":  "Set2",
        "highlight":    "#d94801",
        "neutral":      "#969696",
        "hate_color":   "#e6550d",
        "no_hate_color":"#74c476",
    },
    "C": {
        "name":         "Muted scientific — Nature / IEEE journal style",
        "sequential":   "viridis",
        "diverging":    "coolwarm",
        "categorical":  "tab10",
        "highlight":    "#31a354",
        "neutral":      "#636363",
        "hate_color":   "#e6550d",
        "no_hate_color":"#3182bd",
    },
    "D": {
        "name":         "High contrast — strong accessibility, poster-friendly",
        "sequential":   "YlGnBu",
        "diverging":    "PiYG",
        "categorical":  "Dark2",
        "highlight":    "#7b2d8b",
        "neutral":      "#525252",
        "hate_color":   "#d01c8b",
        "no_hate_color":"#4dac26",
    },
    "E": {
        "name":         "Pastel soft — poster / infographic style",
        "sequential":   "PuBuGn",
        "diverging":    "PRGn",
        "categorical":  "Pastel1",
        "highlight":    "#74c476",
        "neutral":      "#bdbdbd",
        "hate_color":   "#fc8d59",
        "no_hate_color":"#91bfdb",
    },
}

# ─── Change this one line to switch the whole notebook's look ────────────────
ACTIVE_PALETTE = "A"
# ─────────────────────────────────────────────────────────────────────────────

PAL = PALETTES[ACTIVE_PALETTE]
print(f"Active palette : {ACTIVE_PALETTE} — {PAL['name']}")
sns.set_theme(style="whitegrid")

In [ ]:
CONFIG_PATH = "configs/datasets.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

PREPROCESSED_ROOT = cfg["preprocessing"]["output_root"]
TRAINING_ROOT     = cfg["training"]["output_root"]
EVAL_ROOT         = cfg["evaluation"]["output_root"]
EMB_ROOT          = cfg["embedding"]["output_root"]
MAX_LENGTH        = cfg["training"]["max_length"]
EMB_MODEL_SLUG    = cfg["embedding"]["models"][0].replace("/", "_")
TOXIGEN_GROUPS    = cfg["datasets"]["toxigen"]["groups"]

RUSSIAN_TEST_CSV  = os.path.join(PREPROCESSED_ROOT, "russian", "full.csv")
TOXIGEN_TEST_CSV  = os.path.join(PREPROCESSED_ROOT, "toxigen", "test.csv")
EXPERIMENT_NAME   = "main"
OUTPUT_DIR        = os.path.join(EVAL_ROOT, EXPERIMENT_NAME)
PREDICTIONS_DIR   = os.path.join(OUTPUT_DIR, "predictions")
PLOTS_DIR         = os.path.join(OUTPUT_DIR, "plots")
RESULTS_CSV       = os.path.join(OUTPUT_DIR, "results.csv")

os.makedirs(PLOTS_DIR, exist_ok=True)

# ── Run flags — set False to skip recomputation and load cached files ─────────
RUN_ENSEMBLE_SWEEP     = True
RUN_SPECIALIST_MATRIX  = True
RUN_LLM_CATEGORIZATION = False  # flip when notebook 04_01 results are available
# ─────────────────────────────────────────────────────────────────────────────

print("Results CSV    :", RESULTS_CSV)
print("Predictions dir:", PREDICTIONS_DIR)
print("Plots dir      :", PLOTS_DIR)

---
## 1. Bootstrap Confidence Intervals & Statistical Tests

Extends the F1-only CIs already in `results.csv` to **all four metrics**
(F1, balanced accuracy, AUC-ROC, accuracy) and adds **paired bootstrap
p-values** for the key model comparisons.

Requires per-sample prediction CSVs in `outputs/4_evaluation/main/predictions/`
(written by `evaluate_all` after the `comparator.py` update).

In [ ]:
results = pd.read_csv(RESULTS_CSV)
METRICS = ["f1", "balanced_accuracy", "auc_roc", "accuracy"]

ci_rows = []
for _, row in results.iterrows():
    safe      = row["run_name"].replace("/", "_").replace("\\", "_")
    pred_path = os.path.join(PREDICTIONS_DIR, f"{safe}.csv")
    entry     = {"run_name": row["run_name"]}

    if os.path.exists(pred_path):
        preds  = pd.read_csv(pred_path)
        y_true = preds["y_true"].values
        y_prob = preds["y_prob"].values
        for m in METRICS:
            # LLM baselines have no continuous probability — skip AUC
            if m == "auc_roc" and row.get("train_dataset") == "llm_baseline":
                entry[f"{m}_ci_lower"] = np.nan
                entry[f"{m}_ci_upper"] = np.nan
            else:
                ci = bootstrap_ci(y_true, y_prob, metric=m, n_bootstrap=1000)
                entry[f"{m}_ci_lower"] = ci["ci_lower"]
                entry[f"{m}_ci_upper"] = ci["ci_upper"]
    else:
        print(f"[WARN] No predictions file for {row['run_name']!r} — falling back to results.csv CIs")
        for m in METRICS:
            entry[f"{m}_ci_lower"] = row.get(f"{m}_ci_lower", np.nan)
            entry[f"{m}_ci_upper"] = row.get(f"{m}_ci_upper", np.nan)

    ci_rows.append(entry)

ci_df      = pd.DataFrame(ci_rows)
# Merge: prefer newly-computed CIs over any existing columns of the same name
results_ci = results.drop(columns=[c for c in results.columns if c.endswith(("_ci_lower", "_ci_upper"))], errors="ignore")
results_ci = results_ci.merge(ci_df, on="run_name", how="left")

results_ci.to_csv(os.path.join(OUTPUT_DIR, "results_ci.csv"), index=False)
print(f"Saved → {OUTPUT_DIR}/results_ci.csv  ({len(results_ci)} rows)")
print(results_ci[["run_name", "f1", "f1_ci_lower", "f1_ci_upper"]].sort_values("f1", ascending=False).to_string())

In [ ]:
df_plot = results_ci.dropna(subset=["f1"]).sort_values("f1", ascending=True).copy()
df_plot["label"] = df_plot["run_name"].str[-55:]
colors = [PAL["highlight"] if w else PAL["neutral"] for w in df_plot["weighted"].fillna(False)]

fig, ax = plt.subplots(figsize=(10, max(5, len(df_plot) * 0.42)))
ax.barh(df_plot["label"], df_plot["f1"], color=colors, alpha=0.8)

if "f1_ci_lower" in df_plot.columns:
    ax.errorbar(
        df_plot["f1"], df_plot["label"],
        xerr=[df_plot["f1"] - df_plot["f1_ci_lower"], df_plot["f1_ci_upper"] - df_plot["f1"]],
        fmt="none", color="black", capsize=3, linewidth=1,
    )

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=PAL["highlight"], alpha=0.8, label="Density-weighted"),
    Patch(color=PAL["neutral"],   alpha=0.8, label="Unweighted / baseline"),
])
ax.set_xlabel("F1 Score (95% bootstrap CI)")
ax.set_title("Model Comparison — F1 on Russian Annotated Test Set")
ax.axvline(0.5, color="grey", linestyle=":", linewidth=0.8)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "01_f1_ci_forest.png"), dpi=150)
plt.show()

In [ ]:
# ── Inline paired bootstrap p-value (two-sided) ──────────────────────────────
def _paired_bootstrap_pvalue(y_true, y_prob_a, y_prob_b, metric="f1", n=10_000, seed=42):
    """H0: metric(A) == metric(B).  Returns two-sided p-value."""
    _fns = {
        "f1":                lambda yt, yp: f1_score(yt, (yp >= 0.5).astype(int), zero_division=0),
        "auc_roc":           roc_auc_score,
        "balanced_accuracy": lambda yt, yp: balanced_accuracy_score(yt, (yp >= 0.5).astype(int)),
    }
    fn  = _fns[metric]
    rng = np.random.default_rng(seed)
    n_s = len(y_true)
    observed = fn(y_true, y_prob_a) - fn(y_true, y_prob_b)
    count = 0
    for _ in range(n):
        idx  = rng.integers(0, n_s, n_s)
        diff = fn(y_true[idx], y_prob_a[idx]) - fn(y_true[idx], y_prob_b[idx])
        if abs(diff) >= abs(observed):
            count += 1
    return count / n

# ── Define comparison pairs ───────────────────────────────────────────────────
top      = results_ci.dropna(subset=["f1"]).sort_values("f1", ascending=False)
best     = top.iloc[0]["run_name"]
best_uw  = top[top["weighted"] == False].iloc[0]["run_name"]
second   = top.iloc[1]["run_name"]

pairs = [
    (best, best_uw, "Best weighted vs. best unweighted (same arch)"),
    (best, second,  "1st overall vs. 2nd overall"),
]

pval_rows = []
for run_a, run_b, label in pairs:
    for safe, run in [(run_a.replace("/","_").replace("\\","_"), run_a),
                      (run_b.replace("/","_").replace("\\","_"), run_b)]:
        pass  # just to compute safes
    safe_a = run_a.replace("/","_").replace("\\","_")
    safe_b = run_b.replace("/","_").replace("\\","_")
    pa     = os.path.join(PREDICTIONS_DIR, f"{safe_a}.csv")
    pb     = os.path.join(PREDICTIONS_DIR, f"{safe_b}.csv")
    if not (os.path.exists(pa) and os.path.exists(pb)):
        print(f"[SKIP] Missing predictions for: {label}")
        continue
    pred_a = pd.read_csv(pa)
    pred_b = pd.read_csv(pb)
    y_true_shared = pred_a["y_true"].values
    for m in ["f1", "balanced_accuracy"]:
        p = _paired_bootstrap_pvalue(y_true_shared, pred_a["y_prob"].values, pred_b["y_prob"].values, metric=m)
        val_a = results_ci.loc[results_ci.run_name == run_a, m].values[0]
        val_b = results_ci.loc[results_ci.run_name == run_b, m].values[0]
        pval_rows.append({"comparison": label, "metric": m,
                           run_a[:30]: round(val_a, 4), run_b[:30]: round(val_b, 4),
                           "p_value": round(p, 4), "significant (p<0.05)": p < 0.05})

if pval_rows:
    pval_df = pd.DataFrame(pval_rows)
    display(
        pval_df.style
        .format({"p_value": "{:.4f}"})
        .applymap(lambda v: "background-color: #d4edda" if v is True else
                            "background-color: #f8d7da" if v is False else "",
                  subset=["significant (p<0.05)"])
    )
else:
    print("No prediction files found — re-run notebook 04 with the updated comparator.")

---
## 2. Reverse Density: Russian Coverage under ToxiGen

For each Russian annotated sample, we ask: *how well does ToxiGen cover it?*

- **Coverage** (`density_k100_all`): overall log-density of the sample under the combined ToxiGen manifold.
- **Closest group** (`argmax_g density_k100_{g}`): which of the 13 ToxiGen groups is most similar in embedding space.

High-density samples sit comfortably inside the training distribution; low-density samples are novel — the model has never seen anything like them. Cross-referencing with hate/no-hate labels surfaces whether generalisation failures are density-driven.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
DENSITY_K = 100  # K used to compute density (must match a value in configs/datasets.yaml k_values)
# ──────────────────────────────────────────────────────────────────────────────

russian_densities_path = os.path.join(EMB_ROOT, "russian", EMB_MODEL_SLUG, "densities.csv")
russian_labels_path    = RUSSIAN_TEST_CSV

densities = pd.read_csv(russian_densities_path)
labels_df = pd.read_csv(russian_labels_path)[["label"]].rename(columns={"label": "true_label"})

df_rev = pd.concat([labels_df.reset_index(drop=True), densities.reset_index(drop=True)], axis=1)
df_rev["true_label"] = df_rev["true_label"].map({"hate": "hate", "no_hate": "no_hate", "no hate": "no_hate"})

# Per-group density columns for the 13 ToxiGen groups
group_cols = [f"density_k{DENSITY_K}_{g}" for g in TOXIGEN_GROUPS if f"density_k{DENSITY_K}_{g}" in df_rev.columns]
coverage_col = f"density_k{DENSITY_K}_all"

if group_cols:
    df_rev["closest_group"] = df_rev[group_cols].idxmax(axis=1).str.replace(f"density_k{DENSITY_K}_", "", regex=False)
    print(f"Loaded {len(df_rev)} Russian samples | {len(group_cols)} group density columns found")
else:
    print(f"[WARN] No per-group columns for K={DENSITY_K} found in {russian_densities_path}")
    print("Available columns:", [c for c in densities.columns if 'density' in c])

In [ ]:
if coverage_col not in df_rev.columns:
    print(f"[SKIP] Coverage column '{coverage_col}' not found.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: coverage distribution by label
    for label, color in [("hate", PAL["hate_color"]), ("no_hate", PAL["no_hate_color"])]:
        subset = df_rev.loc[df_rev["true_label"] == label, coverage_col].dropna()
        axes[0].hist(subset, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[0].set_xlabel(f"Log-density under ToxiGen (k={DENSITY_K})")
    axes[0].set_ylabel("Density")
    axes[0].set_title("ToxiGen Coverage of Russian Samples")
    axes[0].legend()

    # Right: box per label
    plot_data = df_rev[["true_label", coverage_col]].dropna()
    sns.boxplot(data=plot_data, x="true_label", y=coverage_col, ax=axes[1],
                palette={"hate": PAL["hate_color"], "no_hate": PAL["no_hate_color"]})
    axes[1].set_xlabel("")
    axes[1].set_ylabel(f"Log-density (k={DENSITY_K})")
    axes[1].set_title("Coverage Distribution by Label")

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "02_reverse_density_coverage.png"), dpi=150)
    plt.show()

    for label in ["hate", "no_hate"]:
        vals = df_rev.loc[df_rev["true_label"] == label, coverage_col].dropna()
        print(f"{label:8s}: mean={vals.mean():.2f}  median={vals.median():.2f}  n={len(vals)}")

In [ ]:
if "closest_group" not in df_rev.columns:
    print("[SKIP] Closest-group column not available.")
else:
    counts = (
        df_rev.groupby(["closest_group", "true_label"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["hate", "no_hate"])
        .sort_values("hate", ascending=False)
    )

    fig, ax = plt.subplots(figsize=(11, 4))
    counts.plot.bar(ax=ax, color=[PAL["hate_color"], PAL["no_hate_color"]], alpha=0.85, width=0.7)
    ax.set_xlabel("Closest ToxiGen Group")
    ax.set_ylabel("Number of Russian Samples")
    ax.set_title("Closest ToxiGen Group per Russian Sample (by hate label)")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right")
    ax.legend(title="Label")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "02_reverse_density_groups.png"), dpi=150)
    plt.show()

    total = len(df_rev)
    print("Group assignment summary:")
    print(counts.assign(total=counts.sum(axis=1)).assign(pct=lambda d: (d["total"]/total*100).round(1)).to_string())

---
## 3. Hyperparameter Ablation: K and Embedding Space

All density-weighted runs share the same architecture and training recipe;
the only difference is (K, embedding space). This section turns `results.csv`
into a visual ablation to justify the K=100 / PCA choice made in Section 7.4
of the thesis.

In [ ]:
df_abl = results_ci.dropna(subset=["k", "f1", "space"]).copy()
df_abl["k"] = df_abl["k"].astype(int)

if df_abl.empty:
    print("[SKIP] No density-weighted rows with K and space columns — run full training first.")
else:
    # One facet per base model
    model_slugs = df_abl["model_slug"].unique()
    ncols = len(model_slugs)
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 4), sharey=True)
    if ncols == 1:
        axes = [axes]

    for ax, slug in zip(axes, model_slugs):
        sub = df_abl[df_abl["model_slug"] == slug]
        for space, ls in [("raw", "-"), ("pca", "--")]:
            s = sub[sub["space"] == space].sort_values("k")
            if s.empty:
                continue
            ax.plot(s["k"], s["f1"], marker="o", linestyle=ls,
                    color=PAL["highlight"] if space == "pca" else PAL["neutral"],
                    label=f"space={space}")
            if "f1_ci_lower" in s.columns:
                ax.fill_between(s["k"], s["f1_ci_lower"], s["f1_ci_upper"], alpha=0.15,
                                color=PAL["highlight"] if space == "pca" else PAL["neutral"])
        ax.set_xscale("log")
        ax.set_xticks([5, 100, 1000])
        ax.set_xticklabels(["5", "100", "1000"])
        ax.set_xlabel("K")
        ax.set_ylabel("F1")
        ax.set_title(slug)
        ax.legend()

    fig.suptitle("F1 vs K — raw vs PCA embedding space (95% CI shaded)", y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "03_k_space_ablation.png"), dpi=150, bbox_inches="tight")
    plt.show()

    print("\nBest configuration per model:")
    print(df_abl.loc[df_abl.groupby("model_slug")["f1"].idxmax(), ["model_slug", "k", "space", "f1"]])

---
## 4. Ensemble Normalisation Sweep

Section 6.3.2 of the thesis fixes the normalisation temperature at T=5. This
section sweeps T ∈ {1, 2, 5, 10, 50} to confirm that choice, and adds two
extra baselines — **majority vote** and **uniform weights** — to frame T as a
deliberate hyperparameter rather than an arbitrary constant.

Results are cached to `outputs/4_evaluation/main/ensemble_sweep.csv`.

In [ ]:
# ── Parameters (keep aligned with best K/space from Section 3) ────────────────
ENSEMBLE_K     = 100
ENSEMBLE_SPACE = "pca"
# ─────────────────────────────────────────────────────────────────────────────

TOXIGEN_DENSITIES_CSV = os.path.join(
    cfg["embedding"]["output_root"], "toxigen", EMB_MODEL_SLUG, "densities.csv"
)

SWEEP_CONFIGS = [
    {"voting": "majority",          "norm": "softmax", "temp": 5.0,  "label": "Majority vote"},
    {"voting": "weighted_average",  "norm": "softmax", "temp": 1.0,  "label": "Softmax T=1 (peaky)"},
    {"voting": "weighted_average",  "norm": "softmax", "temp": 2.0,  "label": "Softmax T=2"},
    {"voting": "weighted_average",  "norm": "softmax", "temp": 5.0,  "label": "Softmax T=5 (thesis)"},
    {"voting": "weighted_average",  "norm": "softmax", "temp": 10.0, "label": "Softmax T=10"},
    {"voting": "weighted_average",  "norm": "softmax", "temp": 50.0, "label": "Softmax T=50 (≈uniform)"},
    {"voting": "weighted_average",  "norm": "linear",  "temp": 5.0,  "label": "Linear norm (thesis alt)"},
]

SWEEP_CACHE = os.path.join(OUTPUT_DIR, "ensemble_sweep.csv")

if RUN_ENSEMBLE_SWEEP:
    sweep_rows = []
    for ecfg in SWEEP_CONFIGS:
        print(f"Running: {ecfg['label']} ...")
        m = evaluate_ensemble(
            test_csv              = RUSSIAN_TEST_CSV,
            toxigen_densities_csv = TOXIGEN_DENSITIES_CSV,
            training_root         = TRAINING_ROOT,
            output_dir            = OUTPUT_DIR,
            k                     = ENSEMBLE_K,
            space                 = ENSEMBLE_SPACE,
            voting                = ecfg["voting"],
            model_slug            = None,
            batch_size            = 32,
            max_length            = MAX_LENGTH,
            bootstrap             = True,
            n_bootstrap           = 1000,
            norm_method           = ecfg["norm"],
            norm_temperature      = ecfg["temp"],
        )
        sweep_rows.append({
            "label":             ecfg["label"],
            "voting":            ecfg["voting"],
            "norm":              ecfg["norm"],
            "temp":              ecfg["temp"],
            "f1":                m["f1"],
            "balanced_accuracy": m["balanced_accuracy"],
            "auc_roc":           m["auc_roc"],
            "f1_ci_lower":       m.get("f1_ci_lower"),
            "f1_ci_upper":       m.get("f1_ci_upper"),
        })
    sweep_df = pd.DataFrame(sweep_rows)
    sweep_df.to_csv(SWEEP_CACHE, index=False)
    print(f"Saved sweep → {SWEEP_CACHE}")
else:
    sweep_df = pd.read_csv(SWEEP_CACHE)
    print(f"Loaded cached sweep from {SWEEP_CACHE}")

display(
    sweep_df.sort_values("f1", ascending=False)
    .style.format({"f1": "{:.4f}", "balanced_accuracy": "{:.4f}", "auc_roc": "{:.4f}"})
    .background_gradient(subset=["f1"], cmap=PAL["sequential"])
)

In [ ]:
softmax_df = sweep_df[sweep_df["voting"] == "weighted_average"].copy()

fig, ax = plt.subplots(figsize=(8, 4))

# Softmax T sweep
for norm, color, ls in [("softmax", PAL["highlight"], "-"), ("linear", PAL["neutral"], "--")]:
    sub = softmax_df[softmax_df["norm"] == norm].sort_values("temp")
    if sub.empty:
        continue
    ax.plot(sub["temp"], sub["f1"], marker="o", color=color, linestyle=ls, label=f"norm={norm}")
    if "f1_ci_lower" in sub.columns:
        ax.fill_between(sub["temp"], sub["f1_ci_lower"].astype(float),
                        sub["f1_ci_upper"].astype(float), alpha=0.15, color=color)

# Majority vote as horizontal reference
maj = sweep_df[sweep_df["voting"] == "majority"]
if not maj.empty:
    ax.axhline(maj.iloc[0]["f1"], color="grey", linestyle=":", linewidth=1.2, label="Majority vote")

ax.set_xscale("log")
ax.set_xlabel("Temperature T")
ax.set_ylabel("F1 Score")
ax.set_title(f"Ensemble F1 vs Normalisation Temperature (K={ENSEMBLE_K}, space={ENSEMBLE_SPACE})")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "04_ensemble_temp_sweep.png"), dpi=150)
plt.show()

---
## 5. Cross-group Specialist Evaluation Matrix

Each specialist was trained on one ToxiGen group. We ask: *does a specialist
transfer to other groups, and to Russian hate speech?*

For every (specialist, target group) pair we run inference and record AUC-ROC.
The result is a 13 × 14 heatmap (rows = specialist group, columns = target
group + Russian). High off-diagonal AUC means the specialist generalises;
low values reveal which groups are complementary in the ensemble.

Results cached to `outputs/4_evaluation/main/specialist_matrix.csv`.

In [ ]:
# ── Create per-group ToxiGen test splits ──────────────────────────────────────
BY_GROUP_DIR = os.path.join(PREPROCESSED_ROOT, "toxigen", "by_group")
os.makedirs(BY_GROUP_DIR, exist_ok=True)

toxigen_test = pd.read_csv(TOXIGEN_TEST_CSV)
if "group" not in toxigen_test.columns:
    raise RuntimeError(f"'group' column not found in {TOXIGEN_TEST_CSV}. Re-run notebook 01.")

for group, gdf in toxigen_test.groupby("group"):
    out = os.path.join(BY_GROUP_DIR, f"{group}.csv")
    if not os.path.exists(out):
        gdf[["text", "label"]].to_csv(out, index=False)

print(f"By-group splits ready in {BY_GROUP_DIR}")
for g in TOXIGEN_GROUPS:
    path = os.path.join(BY_GROUP_DIR, f"{g}.csv")
    n = len(pd.read_csv(path)) if os.path.exists(path) else 0
    print(f"  {g:20s}: {n} samples")

In [ ]:
MATRIX_CACHE = os.path.join(OUTPUT_DIR, "specialist_matrix.csv")
label2id     = {"hate": 1, "no_hate": 0, "no hate": 0}

if RUN_SPECIALIST_MATRIX:
    specialists = discover_models(TRAINING_ROOT, specialists=True)
    if not specialists:
        print("[WARN] No specialist models found — run notebook 03_01 first.")
    else:
        test_csvs = {g: os.path.join(BY_GROUP_DIR, f"{g}.csv") for g in TOXIGEN_GROUPS}
        test_csvs["russian"] = RUSSIAN_TEST_CSV

        matrix_rows = []
        total = len(specialists) * len(test_csvs)
        done  = 0
        for spec in specialists:
            spec_group = spec["dataset"].replace("spec_", "")
            for target, csv_path in test_csvs.items():
                done += 1
                if not os.path.exists(csv_path):
                    print(f"  [{done}/{total}] MISSING: {csv_path}")
                    continue
                df_t   = pd.read_csv(csv_path)
                texts  = df_t["text"].astype(str).tolist()
                y_true = df_t["label"].map(label2id).fillna(df_t["label"]).astype(int).values
                try:
                    y_prob = _get_probabilities(spec["model_path"], texts, max_length=MAX_LENGTH)
                    m      = compute_metrics(y_true, y_prob)
                    matrix_rows.append({"specialist": spec_group, "model_slug": spec["model_slug"],
                                        "target": target, **m})
                except Exception as e:
                    print(f"  [{done}/{total}] ERROR {spec_group} → {target}: {e}")

        matrix_df = pd.DataFrame(matrix_rows)
        matrix_df.to_csv(MATRIX_CACHE, index=False)
        print(f"Specialist matrix saved → {MATRIX_CACHE}  ({len(matrix_df)} rows)")
else:
    matrix_df = pd.read_csv(MATRIX_CACHE)
    print(f"Loaded cached specialist matrix ({len(matrix_df)} rows)")

In [ ]:
if 'matrix_df' not in dir() or matrix_df.empty:
    print("[SKIP] No specialist matrix data available.")
else:
    # One heatmap per base model slug found in the matrix
    for slug, sub in matrix_df.groupby("model_slug"):
        pivot = (
            sub.pivot_table(index="specialist", columns="target", values="auc_roc", aggfunc="mean")
            .reindex(index=TOXIGEN_GROUPS, columns=TOXIGEN_GROUPS + ["russian"])
        )

        fig, ax = plt.subplots(figsize=(14, 9))
        sns.heatmap(
            pivot, annot=True, fmt=".2f",
            cmap=PAL["sequential"], vmin=0.4, vmax=1.0,
            linewidths=0.4, annot_kws={"size": 7}, ax=ax,
        )
        ax.set_title(f"Specialist × Target AUC-ROC — {slug}")
        ax.set_xlabel("Target group (test set)")
        ax.set_ylabel("Specialist (trained on)")
        ax.tick_params(axis="x", rotation=40)
        plt.tight_layout()
        safe_slug = slug.replace("/", "_")
        plt.savefig(os.path.join(PLOTS_DIR, f"05_specialist_matrix_{safe_slug}.png"), dpi=150, bbox_inches="tight")
        plt.show()

---
## 6. Calibration Analysis

AUC and F1 measure discrimination; **calibration** measures whether a predicted
probability of 0.7 actually means 70% chance of hate. Good calibration matters
for deployment: operators set a threshold on `p_hate` to control FPR, not on a
fixed 0.5 cutoff.

We plot **reliability diagrams** and report **Expected Calibration Error (ECE)**
for the top-3 models by F1.

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """Uniform-width binning ECE."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)
    return ece

# Load top-3 models by F1
top3_runs = results_ci.dropna(subset=["f1"]).sort_values("f1", ascending=False).head(3)["run_name"].tolist()

cal_records = []
for run_name in top3_runs:
    safe = run_name.replace("/", "_").replace("\\", "_")
    path = os.path.join(PREDICTIONS_DIR, f"{safe}.csv")
    if not os.path.exists(path):
        print(f"[SKIP] No predictions for {run_name!r}")
        continue
    preds  = pd.read_csv(path)
    y_true = preds["y_true"].values
    y_prob = preds["y_prob"].values
    ece    = expected_calibration_error(y_true, y_prob)
    cal_records.append({"run_name": run_name, "ece": ece, "y_true": y_true, "y_prob": y_prob})
    print(f"{run_name[-55:]:55s}  ECE = {ece:.4f}")

In [ ]:
if not cal_records:
    print("[SKIP] No calibration data available.")
else:
    fig, axes = plt.subplots(1, len(cal_records), figsize=(5 * len(cal_records), 4), sharey=True)
    if len(cal_records) == 1:
        axes = [axes]

    for ax, rec in zip(axes, cal_records):
        frac_pos, mean_pred = calibration_curve(rec["y_true"], rec["y_prob"], n_bins=10)
        ax.plot([0, 1], [0, 1], linestyle=":", color="grey", linewidth=1, label="Perfect")
        ax.plot(mean_pred, frac_pos, marker="o", color=PAL["highlight"], linewidth=1.5,
                label=f"ECE = {rec['ece']:.3f}")
        ax.fill_between(mean_pred, frac_pos, mean_pred, alpha=0.15, color=PAL["highlight"])
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Fraction of positives")
        ax.set_title(rec["run_name"][-40:], fontsize=8)
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

    fig.suptitle("Reliability Diagrams — Top-3 Models by F1")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "06_reliability_diagrams.png"), dpi=150)
    plt.show()

    # Add ECE to the extended results table
    ece_map = {r["run_name"]: r["ece"] for r in cal_records}
    results_ci["ece"] = results_ci["run_name"].map(ece_map)
    results_ci.to_csv(os.path.join(OUTPUT_DIR, "results_ci.csv"), index=False)
    print("\nECE added to results_ci.csv")

---
## 7. Qualitative Error Analysis

False positives (predicted hate, actually benign) and false negatives (missed
hate) reveal which linguistic phenomena the model struggles with. We:

1. Split errors from the best model into FP and FN sets.
2. Embed each error with `all-mpnet-base-v2` and cluster into k=8 groups.
3. Display 3 representative examples per cluster for manual inspection.
4. *(Optional)* Run an LLM-structured-output pass to assign each error a
   category (sarcasm, dog whistle, dark humour, …). Gated behind
   `RUN_LLM_CATEGORIZATION` — flip once notebook `04_01` results exist.

In [ ]:
best_run = results_ci.dropna(subset=["f1"]).sort_values("f1", ascending=False).iloc[0]["run_name"]
print("Best model:", best_run)

safe      = best_run.replace("/", "_").replace("\\", "_")
pred_path = os.path.join(PREDICTIONS_DIR, f"{safe}.csv")

if not os.path.exists(pred_path):
    raise FileNotFoundError(
        f"Predictions not found: {pred_path}\n"
        "Re-run notebook 04 with the updated comparator."
    )

preds_df  = pd.read_csv(pred_path)
test_df   = pd.read_csv(RUSSIAN_TEST_CSV)[["text", "label"]].reset_index(drop=True)
errors_df = pd.concat([test_df, preds_df], axis=1)

fp = errors_df[(errors_df["y_true"] == 0) & (errors_df["y_pred"] == 1)].copy()
fn = errors_df[(errors_df["y_true"] == 1) & (errors_df["y_pred"] == 0)].copy()

print(f"False Positives (predicted hate, actual no_hate) : {len(fp)}")
print(f"False Negatives (predicted no_hate, actual hate) : {len(fn)}")

In [ ]:
N_CLUSTERS     = 8
EMBED_MODEL_EA = "sentence-transformers/all-mpnet-base-v2"  # same model used in pipeline

if len(fp) + len(fn) == 0:
    print("[SKIP] No errors to cluster.")
else:
    print(f"Embedding {len(fp) + len(fn)} error samples with {EMBED_MODEL_EA} ...")
    _ea_model  = SentenceTransformer(EMBED_MODEL_EA)
    all_errors = pd.concat([fp.assign(error_type="FP"), fn.assign(error_type="FN")]).reset_index(drop=True)
    embeddings = _ea_model.encode(all_errors["text"].tolist(), show_progress_bar=True,
                                  batch_size=64, normalize_embeddings=True)

    km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")
    all_errors["cluster"] = km.fit_predict(embeddings)
    print(f"K-means done. Cluster sizes:")
    print(all_errors.groupby(["cluster", "error_type"]).size().unstack(fill_value=0).to_string())

In [ ]:
if 'all_errors' not in dir() or all_errors.empty:
    print("[SKIP] Run the clustering cell first.")
else:
    for cluster_id in sorted(all_errors["cluster"].unique()):
        sub = all_errors[all_errors["cluster"] == cluster_id]
        fp_n = (sub["error_type"] == "FP").sum()
        fn_n = (sub["error_type"] == "FN").sum()
        print(f"\n── Cluster {cluster_id}  (FP={fp_n}, FN={fn_n}) " + "─" * 40)
        # Pick 3 samples closest to the centroid
        cluster_embs  = embeddings[all_errors["cluster"].values == cluster_id]
        centroid      = km.cluster_centers_[cluster_id]
        dists         = np.linalg.norm(cluster_embs - centroid, axis=1)
        top3_idx      = np.argsort(dists)[:3]
        for rank, idx in enumerate(top3_idx, 1):
            row = sub.iloc[idx]
            print(f"  [{rank}] [{row['error_type']}] p_hate={row['y_prob']:.3f}  \"{row['text'][:120]}\"")

In [ ]:
# ── Optional: LLM-assisted error categorisation ───────────────────────────────
# Flip RUN_LLM_CATEGORIZATION = True at the top of this notebook once
# the LLM baseline results from notebook 04_01 are available and you have
# an Anthropic or OpenAI API key configured.
#
# Categories mirror thesis lines 1184-1195:
#   implicit_hate | sarcasm | dog_whistle | dehumanising_metaphor |
#   political_criticism | reporting | dark_humor | war_language | other

if RUN_LLM_CATEGORIZATION:
    import anthropic  # pip install anthropic

    CATEGORIES = [
        "implicit_hate", "sarcasm", "dog_whistle", "dehumanising_metaphor",
        "political_criticism", "reporting", "dark_humor", "war_language", "other",
    ]
    CATEGORY_PROMPT = """You are an expert hate-speech annotator.
Classify the following text into EXACTLY ONE category from this list:
{categories}

Text: \"{text}\"

Respond with just the category name, nothing else."""

    client = anthropic.Anthropic()
    llm_cats = []
    for _, row in all_errors.iterrows():
        msg = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=20,
            messages=[{"role": "user", "content": CATEGORY_PROMPT.format(
                categories=", ".join(CATEGORIES), text=row["text"][:300]
            )}],
        )
        cat = msg.content[0].text.strip().lower()
        llm_cats.append(cat if cat in CATEGORIES else "other")

    all_errors["llm_category"] = llm_cats
    cat_summary = all_errors.groupby(["llm_category", "error_type"]).size().unstack(fill_value=0)
    display(cat_summary.style.background_gradient(cmap=PAL["sequential"]))
    all_errors.to_csv(os.path.join(OUTPUT_DIR, "error_analysis.csv"), index=False)
    print(f"Saved error analysis → {OUTPUT_DIR}/error_analysis.csv")
else:
    print("LLM categorisation skipped (RUN_LLM_CATEGORIZATION = False).")
    print("Set it to True at the top of this notebook when ready.")